# Notebook 11: rolling origins and direct horizons

Conjugate direct-horizon robustness of the nested conditional-mean ladder across expanding training windows.

In [1]:
import importlib, shared_utils
importlib.reload(shared_utils)
from shared_utils import *

import numpy as np
import pandas as pd
from statsmodels.stats.multitest import multipletests
import time

QUICK = quick_mode()
selection = load_result("nb0_selection")["selection"]
P = int(selection["p"])
K = int(selection["k"])
assert P == 38 and K == 2

d = load_data(network="geographic")
Y_all = d["Y_full"].to_numpy()
dates_all = pd.to_datetime(d["Y_full"].index)
N = d["N"]
T_all = len(Y_all)

W_geo = knn_sparsify(d["networks"]["geographic"], K)
W_uniform = (np.ones((N, N)) - np.eye(N)) / (N - 1)

HORIZONS = [1, 6, 12]
TRAIN_END_DATES = ["2015-12", "2017-06", "2018-12", "2020-06", "2021-12"]
if QUICK:
    TRAIN_END_DATES = [TRAIN_END_DATES[0], TRAIN_END_DATES[-1]]

train_ends = [
    int(np.flatnonzero(dates_all.to_period("M") == pd.Period(date, "M"))[0])
    for date in TRAIN_END_DATES
]

PRIOR = dict(
    prior_type="minnesota",
    lambda1=MINN["lambda1"], lambda2=MINN["lambda2"],
    lambda3=MINN["lambda3"], rw_centre=MINN["rw_centre"],
)

LADDER = {
    "ar": (W_geo, [0] * P),
    "uniform_s1": (W_uniform, [1] * P),
    "geo_s1": (W_geo, [1] * P),
    "geo_s2": (W_geo, [2] * P),
}

## Direct-horizon forecaster

In [2]:
_TREATMENT_CACHE = {}

def panel_treatment(Y, treatment, upto):
    """Return a panel treatment estimated using rows strictly before upto."""
    key = (treatment, int(upto))
    if key in _TREATMENT_CACHE:
        return _TREATMENT_CACHE[key]

    train = Y[:upto]
    factor = Y.mean(axis=1, keepdims=True)
    if treatment == "raw":
        out = (Y, 0.0, np.zeros_like(factor))
    elif treatment == "factor_adjusted":
        def logml(kappa):
            residual = train - kappa * factor[:upto]
            model = BayesianGNAR(p=P, s=[0] * P, **PRIOR)
            try:
                model.fit(residual, [])
                return model.log_marginal_likelihood()
            except (np.linalg.LinAlgError, FloatingPointError):
                return -np.inf

        coarse = np.linspace(0.0, 1.5, 16)
        values = np.array([logml(x) for x in coarse])
        best = int(np.argmax(values))
        if best in {0, len(coarse)-1}:
            raise RuntimeError("Factor-loading profile optimum lies on the coarse-grid boundary.")
        centre = coarse[best]
        fine = np.linspace(centre - 0.1, centre + 0.1, 41)
        kappa = float(fine[int(np.argmax([logml(x) for x in fine]))])
        out = (Y - kappa * factor, kappa, factor)
    else:
        raise ValueError(treatment)

    _TREATMENT_CACHE[key] = out
    return out

def direct_h_forecast(Y, train_end, horizon, treatment, model_key):
    """Fit one direct-h conjugate specification and forecast all feasible later targets."""
    Wm, stages = LADDER[model_key]
    mode = "ar_only" if max(stages) == 0 else "global_gnar"
    treated, kappa, factor_path = panel_treatment(Y, treatment, train_end + 1)

    X_train, y_train, names = build_design(
        treated[:train_end+1], Wm, p=P, stages=stages, mode=mode, h=horizon
    )
    model = BayesianGNAR(p=P, s=stages, **PRIOR)
    model.fit_design(X_train, y_train, names, N)

    y_true, means, variances, dates = [], [], [], []
    for target in range(train_end + horizon, T_all):
        origin = target - horizon
        Xf = build_forecast_features(
            treated[:origin+1], Wm, p=P, stages=stages, mode=mode
        )
        residual_mean, residual_var = model.predict_from_design(Xf)
        if treatment == "factor_adjusted":
            factor_hat, _ = ar1_forecast(factor_path[:origin+1, 0], horizon=horizon)
            mean = residual_mean + kappa * factor_hat
        else:
            mean = residual_mean

        y_true.append(Y[target])
        means.append(mean)
        variances.append(residual_var)
        dates.append(dates_all[target])

    result = {
        "y_true": np.asarray(y_true), "mean": np.asarray(means),
        "var": np.asarray(variances), "dates": pd.to_datetime(dates),
        "kappa": float(kappa),
    }
    assert result["dates"][0] == dates_all[train_end + horizon]
    return result

## Rolling grid

In [3]:
model_rows, pair_rows = [], []
loss_store = {}
t0 = time.time()

PAIRS = [
    ("uniform_s1_minus_ar", "uniform_s1", "ar"),
    ("geo_s1_minus_uniform", "geo_s1", "uniform_s1"),
    ("geo_s2_minus_geo_s1", "geo_s2", "geo_s1"),
]

for horizon in HORIZONS:
    for train_end_date, train_end in zip(TRAIN_END_DATES, train_ends):
        for treatment in ["raw", "factor_adjusted"]:
            forecasts = {
                key: direct_h_forecast(Y_all, train_end, horizon, treatment, key)
                for key in LADDER
            }

            ref = forecasts["ar"]
            for key, fc in forecasts.items():
                np.testing.assert_allclose(fc["y_true"], ref["y_true"], rtol=0.0, atol=0.0)
                assert fc["dates"].equals(ref["dates"])
                score = score_forecasts(fc["y_true"], fc["mean"], fc["var"], fc["dates"])
                loss = crps_series(fc["y_true"], fc["mean"], fc["var"])
                loss_key = f"{treatment}|{train_end_date}|h{horizon}|{key}"
                loss_store[loss_key] = loss.tolist()
                model_rows.append({
                    "treatment": treatment, "train_end": train_end_date,
                    "horizon": horizon, "model": key, "n_test": len(loss),
                    "CRPS": score["CRPS"], "RMSE": score["RMSE"],
                    "kappa": fc["kappa"], "loss_key": loss_key,
                })

            for pair, a, b in PAIRS:
                fa, fb = forecasts[a], forecasts[b]
                la = crps_series(fa["y_true"], fa["mean"], fa["var"])
                lb = crps_series(fb["y_true"], fb["mean"], fb["var"])
                stat, pval = dm_test(la, lb, h_dm=horizon)
                pair_rows.append({
                    "treatment": treatment, "train_end": train_end_date,
                    "horizon": horizon, "comparison": pair,
                    "n_test": len(la), "mean_loss_difference": float((la-lb).mean()),
                    "gain_first_over_second": float((lb-la).mean()),
                    "dm_stat": stat, "p": pval,
                })

models = pd.DataFrame(model_rows)
pairs = pd.DataFrame(pair_rows)
pairs["p_holm"] = np.nan
pairs["reject_holm"] = False

for treatment in pairs["treatment"].unique():
    for comparison in pairs["comparison"].unique():
        mask = (pairs["treatment"] == treatment) & (pairs["comparison"] == comparison)
        reject, adjusted, _, _ = multipletests(pairs.loc[mask, "p"], alpha=0.05, method="holm")
        pairs.loc[mask, "p_holm"] = adjusted
        pairs.loc[mask, "reject_holm"] = reject

print(f"rolling grid complete in {time.time()-t0:.1f}s")
print(pairs.round(6).to_string(index=False))

rolling grid complete in 38.0s
      treatment train_end  horizon           comparison  n_test  mean_loss_difference  gain_first_over_second   dm_stat        p   p_holm  reject_holm
            raw   2015-12        1  uniform_s1_minus_ar     102             -0.004065                0.004065 -4.486706 0.000019 0.000288         True
            raw   2015-12        1 geo_s1_minus_uniform     102             -0.000504                0.000504 -0.899878 0.370326 1.000000        False
            raw   2015-12        1  geo_s2_minus_geo_s1     102             -0.000667                0.000667 -1.688436 0.094413 1.000000        False
factor_adjusted   2015-12        1  uniform_s1_minus_ar     102             -0.000001                0.000001 -3.431978 0.000870 0.009567         True
factor_adjusted   2015-12        1 geo_s1_minus_uniform     102             -0.000249                0.000249 -1.198125 0.233672 1.000000        False
factor_adjusted   2015-12        1  geo_s2_minus_geo_s1     102

## Save results

In [4]:
summary = {}
for treatment in ["raw", "factor_adjusted"]:
    for comparison in [x[0] for x in PAIRS]:
        sub = pairs[(pairs["treatment"] == treatment) & (pairs["comparison"] == comparison)]
        summary[f"{treatment}|{comparison}"] = {
            "n_cells": int(len(sub)),
            "n_positive_gain": int((sub["gain_first_over_second"] > 0).sum()),
            "n_p_below_0_05": int((sub["p"] < 0.05).sum()),
            "n_holm_significant": int(sub["reject_holm"].sum()),
        }

save_result("nb11_rolling_origins", {
    "p": P, "network": "geographic", "k": K,
    "horizons": HORIZONS, "train_end_dates": TRAIN_END_DATES,
    "models": models.to_dict(orient="records"),
    "pairwise": pairs.to_dict(orient="records"),
    "monthly_losses": loss_store,
    "summary": summary,
    "factor_loading": "profiled once per training panel under an AR-only one-step conjugate model",
    "factor_forecast": "AR(1) with intercept from information available at each origin",
    "config": run_config(p=P, stages=[2] * P, network="geographic", k=K,
                         max_stage=2, purpose="rolling_direct_horizon"),
    "quick": QUICK,
})
print("saved nb11_rolling_origins")

saved nb11_rolling_origins
